# Complete Fix Run — Phase 1 Retrieval + Phase 3 Guardrail

One notebook, one consistent project folder (`/content/QuranicRAG`), run top to bottom. This fixes the directory-confusion issues from before by setting up ONE place everything lives, from the very first cell.

**Order matters:**
1. Set up the project folder (once, cleanly)
2. Fix Phase 1 (AyaTEC-augmented retraining)
3. Rebuild the retrieval index on the improved model
4. Apply all 4 Phase 3 fixes (E3, E4, E6 + expanded queries)
5. Run the full pipeline and compare before/after

**Before running:** Runtime → Change runtime type → GPU (T4) → Save.


## 0. Set up ONE project folder — everything lives here, nothing nested

In [ ]:
import os

PROJECT_ROOT = "/content/QuranicRAG"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/src", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/quranNLP/shared/data", exist_ok=True)

%cd {PROJECT_ROOT}
print("Working directory locked in at:", os.getcwd())


/content/QuranicRAG
Working directory locked in at: /content/QuranicRAG


**Rule for the rest of this notebook: every cell either starts with `%cd /content/QuranicRAG` or assumes you never left it.** If anything ever looks wrong, run this to reset:
```python
%cd /content/QuranicRAG
```


## 1. Confirm GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - enable GPU in Runtime settings")


CUDA available: True
Device: Tesla T4


## 2. Mount Drive and locate your existing Phase 1 + Phase 2 outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PHASE1_DIR = "/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output"
ROMA_DIR = "/content/drive/MyDrive/Phase2_Project/Roma_output"

print("Phase 1 outputs:")
!ls "{PHASE1_DIR}"
print("\nRoma's Phase 2 outputs:")
!ls "{ROMA_DIR}"


Mounted at /content/drive
Phase 1 outputs:
b5_real_finetuned  data  index	outputs

Roma's Phase 2 outputs:
data  quranNLP	src


## 3. Upload ALL your code files in ONE go

Select every file below together (Ctrl+click / Cmd+click to multi-select in the file picker):
- `a1_data_acquisition.py` (if needed for verse text - optional, script re-downloads it directly)
- `b1_load_model.py`, `b2_finetune_harness.py`, `b3_test_harness.py`, `b4_build_real_triplets.py`, `b5_finetune_and_benchmark.py`, `b6_build_index_and_retrieval_api.py`, `results_utils.py`
- `c1_cross_lingual_fallback.py`, `c3_query_refinement.py`
- `d1_agent_loop.py`, `d2_cot_prompting.py`, `d3_sufficiency_and_fallback.py`
- `e1_triple_extraction.py`, `e2_knowledge_graph.py`, `e3_structural_alignment.py`, `e4_zero_hallucination_guardrail.py`, `e5_phase2_integration.py`, `e6_retrieval_quality_diagnostic.py`
- `augment_training_with_ayatec.py`

Missing one is fine - you'll get a clear error naming exactly which file is missing when it's needed, rather than a confusing path error.


In [ ]:
%cd /content/QuranicRAG
from google.colab import files
import shutil

print("Select all your .py files together")
uploaded = files.upload()

for filename in list(uploaded.keys()):
    shutil.move(filename, f"src/{filename}")

print(f"\nPlaced {len(uploaded)} files in src/:")
print(sorted(os.listdir("src")))


/content/QuranicRAG
Select all your .py files together


Saving a2_tafsir_acquisition_arabic.py to a2_tafsir_acquisition_arabic.py
Saving a3_preprocess_pipeline_fixed.py to a3_preprocess_pipeline_fixed.py
Saving augment_training_with_ayatec.py to augment_training_with_ayatec.py
Saving b1_load_model.py to b1_load_model.py
Saving b2_finetune_harness.py to b2_finetune_harness.py
Saving b3_test_harness.py to b3_test_harness.py
Saving b4_build_real_triplets.py to b4_build_real_triplets.py
Saving b5_finetune_and_benchmark.py to b5_finetune_and_benchmark.py
Saving b6_build_index_and_retrieval_api.py to b6_build_index_and_retrieval_api.py
Saving build_final_dataset.py to build_final_dataset.py
Saving c1_cross_lingual_fallback.py to c1_cross_lingual_fallback.py
Saving c2_sufficiency_labels.py to c2_sufficiency_labels.py
Saving c3_query_refinement.py to c3_query_refinement.py
Saving d1_agent_loop.py to d1_agent_loop.py
Saving d2_cot_prompting.py to d2_cot_prompting.py
Saving d3_sufficiency_and_fallback.py to d3_sufficiency_and_fallback.py
Saving e1_tr

## 4. Copy Phase 1's model/index and Roma's C1/C2/C3 outputs into the project folder

In [ ]:
%cd /content/QuranicRAG
import shutil

shutil.copytree(f"{PHASE1_DIR}/b5_real_finetuned", "b5_real_finetuned", dirs_exist_ok=True)
shutil.copytree(f"{PHASE1_DIR}/index", "index_v1", dirs_exist_ok=True)  # keep the ORIGINAL index as v1, for before/after comparison

# B4's original training data, needed as input to the AyaTEC augmentation
for fname in ["real_training_pairs.json", "real_training_triplets.json"]:
    src_path = f"{PHASE1_DIR}/data/{fname}"
    if os.path.exists(src_path):
        shutil.copy(src_path, f"data/{fname}")
        print(f"Copied {fname}")
    else:
        print(f"[MISSING] {fname} not found at {src_path} - will need to run B4 fresh (see cell below)")

# Roma's C1/C2/C3 outputs
shutil.copy(f"{ROMA_DIR}/quranNLP/shared/data/ayatec_records.json", "quranNLP/shared/data/ayatec_records.json")
shutil.copy(f"{ROMA_DIR}/quranNLP/shared/data/squad_v2_sample.json", "quranNLP/shared/data/squad_v2_sample.json")
shutil.copy(f"{ROMA_DIR}/data/sufficiency_labels.json", "data/sufficiency_labels.json")

print("\nDone. Contents of data/:", os.listdir("data"))


/content/QuranicRAG
Copied real_training_pairs.json
Copied real_training_triplets.json

Done. Contents of data/: ['real_training_pairs.json', 'real_training_triplets.json', 'sufficiency_labels.json']


In [ ]:
%cd /content/QuranicRAG
import os
print(os.path.exists("src/b2_finetune_harness.py"))
print(sorted(os.listdir("src")))

/content/QuranicRAG
True
['a2_tafsir_acquisition_arabic.py', 'a3_preprocess_pipeline_fixed.py', 'augment_training_with_ayatec.py', 'b1_load_model.py', 'b2_finetune_harness.py', 'b3_test_harness.py', 'b4_build_real_triplets.py', 'b5_finetune_and_benchmark.py', 'b6_build_index_and_retrieval_api.py', 'build_final_dataset.py', 'c1_cross_lingual_fallback.py', 'c2_sufficiency_labels.py', 'c3_query_refinement.py', 'd1_agent_loop.py', 'd2_cot_prompting.py', 'd3_sufficiency_and_fallback.py', 'e1_triple_extraction.py', 'e2_knowledge_graph.py', 'e3_structural_alignment.py', 'e4_zero_hallucination_guardrail.py', 'e5_phase2_integration.py', 'e6_retrieval_quality_diagnostic.py']


## 5. If B4's original training data was missing above, run B4 fresh

Skip this cell if the previous cell already found and copied both files.

In [ ]:
%cd /content/QuranicRAG
import os
if not (os.path.exists("data/real_training_pairs.json") and os.path.exists("data/real_training_triplets.json")):
    # B4 needs Member A's final_cross_reference_index.csv - copy it in if you have it saved,
    # otherwise upload it now
    os.makedirs("quranNLP/shared/data", exist_ok=True)
    if not os.path.exists("quranNLP/shared/data/final_cross_reference_index.csv"):
        from google.colab import files
        print("Select final_cross_reference_index.csv")
        uploaded_csv = files.upload()
        import shutil
        shutil.move(list(uploaded_csv.keys())[0], "quranNLP/shared/data/final_cross_reference_index.csv")
    !python src/b4_build_real_triplets.py
else:
    print("B4 data already present - skipping.")


/content/QuranicRAG
B4 data already present - skipping.


---
# PART A — Fix Phase 1 Retrieval Quality


## 6. Install dependencies

In [ ]:
%cd /content/QuranicRAG
!pip install -q sentence-transformers hnswlib pandas rapidfuzz pyarabic networkx requests groq


/content/QuranicRAG
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.5 MB/s eta 0:00:00


## 7. Run the AyaTEC augmentation

In [ ]:
%cd /content/QuranicRAG
!python src/augment_training_with_ayatec.py


/content/QuranicRAG
[OK] Loaded 207 real AyaTEC question records.
[STEP] Downloading Quran text to resolve verse text for AyaTEC's answers...
[OK] Resolved 6236 verses.
[OK] Built 2455 (question, verse) pairs from AyaTEC.
[OK] Built 2455 triplets with cross-topic negatives.

Topic category coverage added by AyaTEC augmentation:
  قصص الأنبياء: 604 pairs
  محمد (ص): 297 pairs
  الأمم السابقة: 293 pairs
  أحكام الإسلام: 279 pairs
  الكون ومخلوقات الله: 211 pairs
  الغيب: 164 pairs
  الإيمان والمؤمنون: 135 pairs
  جهاد ومعارك وحروب: 134 pairs
  عبادات: 125 pairs
  الإنسان: 58 pairs
  غير مصنف /أخرى: 54 pairs
  القرآن الكريم: 49 pairs
  مفاسد وموبقات: 45 pairs
  الحياة الدنيا: 7 pairs

[SAVED] data/combined_training_pairs.json (12358 tafsir-derived + 2455 AyaTEC-derived = 14813 total)
[SAVED] data/combined_training_triplets.json (12358 tafsir-derived + 2455 AyaTEC-derived = 14813 total)

[RESULT] Augmentation complete. Re-run B5 fine-tuning pointed at:
  data/combined_training_pairs.json
 

## 8. Retrain the embedding model on the combined (tafsir + AyaTEC) data

In [ ]:
import sys
sys.path.insert(0, "/content/QuranicRAG/src")


import json
from b2_finetune_harness import HarnessConfig, run_finetuning

with open("data/combined_training_pairs.json", encoding="utf-8") as f:
    pairs = json.load(f)
with open("data/combined_training_triplets.json", encoding="utf-8") as f:
    triplets = json.load(f)

config = HarnessConfig(
    base_model_name="Omartificial-Intelligence-Space/GATE-AraBert-v1",
    output_dir="./b5_real_finetuned_v2",
    num_train_epochs=2,
    per_device_train_batch_size=64,
    max_seq_length=64,
    matryoshka_dims=[768, 256, 64],
    use_fp16=True,
)
run_finetuning(pairs, triplets, config)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/8.66k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[OK] Fine-tuned model saved to: ./b5_real_finetuned_v2


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
)

## 9. Rebuild the retrieval index on the improved model

In [ ]:
from google.colab import files
print("Select final_cross_reference_index.csv")
uploaded_csv = files.upload()
shutil.move(list(uploaded_csv.keys())[0], "quranNLP/shared/data/final_cross_reference_index.csv")

Select final_cross_reference_index.csv


Saving final_cross_reference_index.csv to final_cross_reference_index.csv


'quranNLP/shared/data/final_cross_reference_index.csv'

In [ ]:
print(os.path.exists("quranNLP/shared/data/final_cross_reference_index.csv"))


True


In [ ]:
!python src/b6_build_index_and_retrieval_api.py --model-path ./b5_real_finetuned_v2

Loading weights: 100% 199/199 [00:00<00:00, 5274.35it/s]
[OK] Built 12472 retrievable entries (6236 verses, 6236 tafsir passages).
Batches: 100% 390/390 [01:10<00:00,  5.51it/s]
[OK] Built HNSW index: 12472 items, dim=768
[SAVED] Index and metadata written to index/

Retrieval verification:

  Query: الرحمة والمغفرة
    [verse] 80:14 (sim=0.828): مرفوعة مطهرة...
    [verse] 56:94 (sim=0.825): وتصلية جحيم...
    [verse] 93:1 (sim=0.824): والضحىٰ...

  Query: الصبر على البلاء
    [verse] 74:7 (sim=0.765): ولربك فاصبر...
    [verse] 89:27 (sim=0.760): يـٰايتها النفس المطمئنة...
    [verse] 53:38 (sim=0.747): الا تزر وازرة وزر اخرىٰ...

[PASS] Retrieval returned results with provenance metadata.

[RESULT] B6 vector index + retrieval API PASSED verification.
Ready for Phase 2's agentic retrieval loop to call RetrievalAPI.retrieve(query, top_k).


---
# PART B — Apply All Phase 3 Fixes


## 10. Load the IMPROVED retrieval API (v2 model + v2 index)

In [ ]:
%cd /content/QuranicRAG
import sys
sys.path.insert(0, "src")

from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI

model = SentenceTransformer("./b5_real_finetuned_v2")
index, entries = load_index(dim=model.get_sentence_embedding_dimension(), out_dir="index")
retrieval_api = RetrievalAPI(model, index, entries)
print(f"Loaded IMPROVED retrieval API: {len(entries)} indexed entries.")


/content/QuranicRAG


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded IMPROVED retrieval API: 12472 indexed entries.


/tmp/ipykernel_1360/216369931.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  index, entries = load_index(dim=model.get_sentence_embedding_dimension(), out_dir="index")


## 11. Fix 2 — Run the retrieval quality diagnostic (before comparing to old results)

In [ ]:
%cd /content/QuranicRAG
from e6_retrieval_quality_diagnostic import diagnose_retrieval_quality, print_retrieval_quality_report

# Fix 1: the expanded 25-query set
test_queries = [
    "ما فوائد الصبر في القرآن", "من هو النبي المعروف بالصبر", "ماذا يقول القرآن عن الرحمة",
    "التوبة والاستغفار", "العدل في الإسلام", "الشكر لله", "الخوف من الله",
    "الايمان بالغيب", "الصدق في القول", "بر الوالدين",
    "من هو النبي المعروف بالحكمة", "ما حكم الربا في الإسلام", "أهمية الصلاة في القرآن",
    "قصة موسى مع فرعون", "من هو النبي الذي ابتلعه الحوت", "الجنة والنار في القرآن",
    "معنى التقوى", "أحكام الزكاة", "الحلال والحرام", "من هو خاتم الأنبياء",
    "قصة آدم وحواء", "الوصية بالإحسان إلى الجار", "معنى التوكل على الله",
    "قصة أصحاب الكهف", "أهمية العلم في الإسلام",
]

reports = diagnose_retrieval_quality(retrieval_api, test_queries, low_quality_threshold=0.5)
print_retrieval_quality_report(reports)


/content/QuranicRAG

RETRIEVAL QUALITY DIAGNOSTIC

[OK] ما فوائد الصبر في القرآن
  top_similarity=0.593, avg_similarity=0.589
  retrieved: ['62:1', '62:2', '62:3', '62:4', '37:145']

[OK] من هو النبي المعروف بالصبر
  top_similarity=0.657, avg_similarity=0.657
  retrieved: ['37:123', '37:124', '37:125', '37:126', '37:128']

[OK] ماذا يقول القرآن عن الرحمة
  top_similarity=0.580, avg_similarity=0.579
  retrieved: ['62:1', '62:2', '62:3', '62:4', '60:2']

[OK] التوبة والاستغفار
  top_similarity=0.771, avg_similarity=0.771
  retrieved: ['51:1', '51:2', '51:3', '51:7', '51:13']

[OK] العدل في الإسلام
  top_similarity=0.657, avg_similarity=0.650
  retrieved: ['62:1', '62:2', '62:3', '62:4', '57:1']

[OK] الشكر لله
  top_similarity=0.846, avg_similarity=0.837
  retrieved: ['92:6', '91:15', '69:21', '101:7', '53:25']

[OK] الخوف من الله
  top_similarity=0.852, avg_similarity=0.825
  retrieved: ['91:15', '74:47', '81:21', '84:16', '56:95']

[OK] الايمان بالغيب
  top_similarity=0.858, avg_simila

## 12. Set up your LLM (Groq via Colab Secrets)

In [ ]:
%cd /content/QuranicRAG
import time
from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def call_llm_with_retry(prompt: str, max_retries: int = 3) -> str:
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=1000,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < max_retries - 1:
                wait_time = 15 * (attempt + 1)
                print(f"[RATE LIMIT] Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise

def call_llm(prompt: str) -> str:
    return call_llm_with_retry(prompt)

print("groq_client and call_llm ready.")


/content/QuranicRAG
groq_client and call_llm ready.


## 13. Build the complete TheologicalAgent (Phase 2, on the IMPROVED retrieval)

In [ ]:
%cd /content/QuranicRAG
import json
from c1_cross_lingual_fallback import CrossLingualLookup
from d3_sufficiency_and_fallback import TheologicalAgent, SufficiencyScorer, CrossLingualFallback
from d1_agent_loop import AgentConfig
from c3_query_refinement import refine_query

with open("quranNLP/shared/data/ayatec_records.json", encoding="utf-8") as f:
    ayatec_records = json.load(f)
with open("quranNLP/shared/data/squad_v2_sample.json", encoding="utf-8") as f:
    squad_records = json.load(f)
lookup = CrossLingualLookup(ayatec_records, squad_records)
fallback = CrossLingualFallback(lookup_fn=lookup)

with open("data/sufficiency_labels.json", encoding="utf-8") as f:
    labeled_examples = json.load(f)
scorer = SufficiencyScorer()
scorer.calibrate(labeled_examples)

theological_agent = TheologicalAgent(
    retrieval_api=retrieval_api,
    call_llm_fn=call_llm,
    sufficiency_scorer=scorer,
    fallback=fallback,
    refine_fn=refine_query,
    config=AgentConfig(max_iterations=3, verbose=False),
    max_chars_per_context_item=400,
)
print("TheologicalAgent assembled on the IMPROVED (v2) retrieval index.")


/content/QuranicRAG
[OK] Calibrated sufficiency threshold: 0.522 (accuracy=100.0% on 200 labeled examples)
TheologicalAgent assembled on the IMPROVED (v2) retrieval index.


## 14. Run all 25 queries through the Phase 3 guardrail — Fix 4 applied (partial verification allowed)

In [ ]:
%cd /content/QuranicRAG
from e5_phase2_integration import verify_theological_agent_answer
from e4_zero_hallucination_guardrail import GuardrailConfig, print_guardrail_result

guardrail_results = []
for q in test_queries:
    print(f"\n{'='*70}\nQUERY: {q}\n{'='*70}")
    result = verify_theological_agent_answer(
        theological_agent, q,
        refine_query_fn=refine_query,
        guardrail_config=GuardrailConfig(
            similarity_threshold=65.0,
            max_refinement_attempts=2,
            max_acceptable_mismatch=0.2,   # Fix 4: accept up to 20% unaligned claims
            verbose=True,
        ),
    )
    print_guardrail_result(result)
    guardrail_results.append({
        "query": q,
        "verified": result.verified,
        "partially_verified": result.partially_verified,
        "stopped_reason": result.stopped_reason,
        "attempts": len(result.attempts),
        "final_mismatch_score": result.final_mismatch_score,
        "final_response": result.final_response,
    })


/content/QuranicRAG

QUERY: ما فوائد الصبر في القرآن

[GUARDRAIL ATTEMPT 1] Query: 'ما فوائد الصبر في القرآن'
[GUARDRAIL] Model reported insufficient context - not a hallucination, but not answerable from this retrieval either.
[C3] Refinement strategy used: synonym_expansion

[GUARDRAIL ATTEMPT 2] Query: 'ما فوائد الصبر في القرآن ثبات احتساب تحمل'

Mismatch score: 0.643 (MISMATCH DETECTED)
Aligned: 5/14
  [OK] (sim=68.3) ('التعريف: الصبر هو إحدى الفضائل التي يشدد عليها', 'في', 'القرآن الكريم')
  [MISMATCH] (sim=63.1) ('وهو يعني الثبات', 'على', 'الحق والاستقامة في مواجهة التحديات والصعوبات')
      closest context match: ('فوضع رسول الله ﷺ يده', 'على', 'سلمان ثم قال: "لو كان الإيمان عند الثريا لناله رجال -أو: رجل-من هؤلاء"')
  [MISMATCH] (sim=60.0) ('الreasoning: يشير القرآن الكريم', 'إلى', 'أهمية الصبر في مواجهة التحديات والصعوبات')
      closest context match: ('والنهي عما يقربهم', 'إلى', 'النار وسخط الله')
  [MISMATCH] (sim=60.0) ('حيث يذكر', 'في', 'الآية [2:286] أن الصبر هو أحد الفض

## 15. Save everything to Drive

In [ ]:
%cd /content/QuranicRAG
import json

DEST = "/content/drive/MyDrive/Phase3_Project/guardrail_output_v2"
!mkdir -p "{DEST}"

with open("phase3_guardrail_results_v2.json", "w", encoding="utf-8") as f:
    json.dump(guardrail_results, f, ensure_ascii=False, indent=2)

!cp phase3_guardrail_results_v2.json "{DEST}/"
!cp -r src "{DEST}/"
!cp -r b5_real_finetuned_v2 "{DEST}/" 2>/dev/null
!cp -r index_v2 "{DEST}/" 2>/dev/null

print(f"Saved to: {DEST}")


/content/QuranicRAG
Saved to: /content/drive/MyDrive/Phase3_Project/guardrail_output_v2


## 16. Summary — before vs. after

Quick tally so you can see the improvement at a glance.

In [ ]:
verified = sum(1 for r in guardrail_results if r["verified"] and not r["partially_verified"])
partial = sum(1 for r in guardrail_results if r["partially_verified"])
insufficient = sum(1 for r in guardrail_results if r["stopped_reason"] == "insufficient_info")
rejected = sum(1 for r in guardrail_results if not r["verified"] and r["stopped_reason"] == "max_attempts_reached")

print(f"Total queries: {len(guardrail_results)}")
print(f"  Fully verified:      {verified}")
print(f"  Partially verified:  {partial}")
print(f"  Insufficient info:   {insufficient}")
print(f"  Rejected (mismatch): {rejected}")


Total queries: 25
  Fully verified:      0
  Partially verified:  1
  Insufficient info:   12
  Rejected (mismatch): 12


In [ ]:

%cd /content/QuranicRAG
import json

with open("phase3_guardrail_results_v2.json", encoding="utf-8") as f:
    results = json.load(f)

for r in sorted(results, key=lambda x: (x["final_mismatch_score"] is None, x["final_mismatch_score"])):
    score = r["final_mismatch_score"]
    score_str = f"{score:.3f}" if score is not None else "N/A"
    print(f"{score_str:>6}  {r['stopped_reason']:<22}  {r['query']}")

/content/QuranicRAG
 0.133  verified                من هو النبي المعروف بالحكمة
 0.250  max_attempts_reached    معنى التقوى
 0.467  max_attempts_reached    ماذا يقول القرآن عن الرحمة
 0.600  max_attempts_reached    الايمان بالغيب
 0.643  max_attempts_reached    ما فوائد الصبر في القرآن
 0.714  max_attempts_reached    من هو النبي المعروف بالصبر
 0.765  max_attempts_reached    التوبة والاستغفار
 0.769  max_attempts_reached    الصدق في القول
 0.778  max_attempts_reached    قصة موسى مع فرعون
 0.789  max_attempts_reached    معنى التوكل على الله
 0.800  max_attempts_reached    الشكر لله
 0.917  max_attempts_reached    الجنة والنار في القرآن
 1.000  max_attempts_reached    الخوف من الله
   N/A  insufficient_info       العدل في الإسلام
   N/A  insufficient_info       بر الوالدين
   N/A  insufficient_info       ما حكم الربا في الإسلام
   N/A  insufficient_info       أهمية الصلاة في القرآن
   N/A  insufficient_info       من هو النبي الذي ابتلعه الحوت
   N/A  insufficient_info       أحكام الزكاة


In [ ]:
%cd /content/QuranicRAG
index_v1_obj, entries_v1 = load_index(dim=model.get_sentence_embedding_dimension(), out_dir="index_v1")
retrieval_api_v1 = RetrievalAPI(model, index_v1_obj, entries_v1)

from e6_retrieval_quality_diagnostic import diagnose_retrieval_quality

print("=== OLD (v1) retrieval ===")
reports_v1 = diagnose_retrieval_quality(retrieval_api_v1, test_queries)
print("\n=== NEW (v2) retrieval ===")
reports_v2 = diagnose_retrieval_quality(retrieval_api, test_queries)

for r1, r2 in zip(reports_v1, reports_v2):
    print(f"{r1.query[:30]:<32} v1={r1.top_similarity:.3f}  v2={r2.top_similarity:.3f}  "
          f"{'IMPROVED' if r2.top_similarity > r1.top_similarity else 'worse/same'}")

/content/QuranicRAG
=== OLD (v1) retrieval ===


/tmp/ipykernel_1360/2223043484.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  index_v1_obj, entries_v1 = load_index(dim=model.get_sentence_embedding_dimension(), out_dir="index_v1")



=== NEW (v2) retrieval ===
ما فوائد الصبر في القرآن         v1=0.558  v2=0.593  IMPROVED
من هو النبي المعروف بالصبر       v1=0.671  v2=0.657  worse/same
ماذا يقول القرآن عن الرحمة       v1=0.612  v2=0.580  worse/same
التوبة والاستغفار                v1=0.789  v2=0.771  worse/same
العدل في الإسلام                 v1=0.695  v2=0.657  worse/same
الشكر لله                        v1=0.788  v2=0.846  IMPROVED
الخوف من الله                    v1=0.744  v2=0.852  IMPROVED
الايمان بالغيب                   v1=0.823  v2=0.858  IMPROVED
الصدق في القول                   v1=0.810  v2=0.846  IMPROVED
بر الوالدين                      v1=0.707  v2=0.663  worse/same
من هو النبي المعروف بالحكمة      v1=0.614  v2=0.624  IMPROVED
ما حكم الربا في الإسلام          v1=0.574  v2=0.565  worse/same
أهمية الصلاة في القرآن           v1=0.566  v2=0.556  worse/same
قصة موسى مع فرعون                v1=0.792  v2=0.825  IMPROVED
من هو النبي الذي ابتلعه الحوت    v1=0.588  v2=0.633  IMPROVED
الجنة والنار في القرآن      